In [ ]:
import os
import sys

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

from pyspark.sql.functions import *
from delta.tables import DeltaTable

# Use the Python executable from the active environment
python_path = sys.executable

os.environ["PYSPARK_PYTHON"] = python_path
os.environ["PYSPARK_DRIVER_PYTHON"] = python_path

# Prefer local loopback for Spark networking
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

builder = SparkSession.builder \
    .appName("PySparkDemo") \
    .master("local[*]") \
    .config("spark.pyspark.python", python_path) \
    .config("spark.pyspark.driver.python", python_path) \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    ) \
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
spark = configure_spark_with_delta_pip(builder).getOrCreate()
print("SparkSession created successfully.")

In [ ]:
delta_path = os.path.abspath(
    "output/delta/schema_demo/retail_sales"
)

print(delta_path)

In [ ]:
# Disable automatic schema evolution
spark.conf.set(
    "spark.databricks.delta.schema.autoMerge.enabled",
    "false"
)

In [ ]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    DoubleType
)
sales_schema = StructType([
    StructField("SalesID", IntegerType(), False),
    StructField("ProductName", StringType(), True),
    StructField("Category", StringType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("Price", DoubleType(), True)
])

In [ ]:
sales_data = [
    (101, "Laptop", "Electronics", 2, 55000.0),
    (102, "Mobile", "Electronics", 3, 25000.0),
    (103, "Chair", "Furniture", 5, 3500.0),
    (104, "Table", "Furniture", 2, 8000.0),
    (105, "Shoes", "Fashion", 4, 2500.0)
]

sales_df = spark.createDataFrame(
    sales_data,
    sales_schema
)

sales_df.show()

sales_df.printSchema()

In [ ]:
sales_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(delta_path)

In [ ]:
delta_df = spark.read \
    .format("delta") \
    .load(delta_path)

delta_df.show()

delta_df.printSchema()

In [ ]:
new_sales_data = [
    (106, "Keyboard", "Electronics", 3, 1500.0),
    (107, "Mouse", "Electronics", 5, 800.0)
]

new_sales_df = spark.createDataFrame(
    new_sales_data,
    sales_schema
)

new_sales_df.write \
    .format("delta") \
    .mode("append") \
    .save(delta_path)

In [ ]:
spark.read \
    .format("delta") \
    .load(delta_path) \
    .orderBy("SalesID") \
    .show()

In [ ]:
invalid_schema = StructType([
    StructField("SalesID", IntegerType(), False),
    StructField("ProductName", StringType(), True),
    StructField("Category", StringType(), True),
    StructField("Quantity", StringType(), True),
    StructField("Price", DoubleType(), True)
])

In [ ]:
invalid_data = [
    (108, "Monitor", "Electronics", "Three", 15000.0)
]

invalid_df = spark.createDataFrame(
    invalid_data,
    invalid_schema
)

invalid_df.show()

invalid_df.printSchema()

In [ ]:
try:
    invalid_df.write \
        .format("delta") \
        .mode("append") \
        .save(delta_path)

except Exception as e:
    print("Schema Enforcement Error:")
    print(str(e))

In [ ]:
delta_df = spark.read \
    .format("delta") \
    .load(delta_path)

print("Total Records:", delta_df.count())

In [ ]:
source_data = [
    (108, "Monitor", "Electronics", "3", 15000.0),
    (109, "Printer", "Electronics", "2", 12000.0)
]

source_df = spark.createDataFrame(
    source_data,
    invalid_schema
)

In [ ]:
corrected_df = source_df.withColumn(
    "Quantity",
    expr("try_cast(Quantity AS INT)")
)

corrected_df.printSchema()

In [ ]:
valid_df = corrected_df.filter(
    col("Quantity").isNotNull()
)

rejected_df = corrected_df.filter(
    col("Quantity").isNull()
)

rejected_df.show()

In [ ]:
valid_df.write \
    .format("delta") \
    .mode("append") \
    .save(delta_path)

In [ ]:
delta_df = spark.read \
    .format("delta") \
    .load(delta_path)

print("Total Records:", delta_df.count())

In [ ]:
payment_schema = StructType([
    StructField("SalesID", IntegerType(), False),
    StructField("ProductName", StringType(), True),
    StructField("Category", StringType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("Price", DoubleType(), True),
    StructField("PaymentMode", StringType(), True)
])

In [ ]:
payment_data = [
    (110, "Tablet", "Electronics", 2, 22000.0,
     "Credit Card"),

    (111, "Sofa", "Furniture", 1, 35000.0,
     "UPI")
]

payment_df = spark.createDataFrame(
    payment_data,
    payment_schema
)

payment_df.show(truncate=False)

In [ ]:
try:
    payment_df.write \
        .format("delta") \
        .mode("append") \
        .save(delta_path)

except Exception as e:
    print("Schema Mismatch Error:")
    print(str(e))

In [ ]:
payment_df.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .save(delta_path)

In [ ]:
evolved_df = spark.read \
    .format("delta") \
    .load(delta_path)

evolved_df.printSchema()

evolved_df.orderBy("SalesID").show(truncate=False)

In [ ]:
spark.conf.set(
    "spark.databricks.delta.schema.autoMerge.enabled",
    "true"
)

In [ ]:
discount_schema = StructType([
    StructField("SalesID", IntegerType(), False),
    StructField("ProductName", StringType(), True),
    StructField("Category", StringType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("Price", DoubleType(), True),
    StructField("PaymentMode", StringType(), True),
    StructField("Discount", DoubleType(), True)
])

In [ ]:
discount_data = [
    (112, "Smart TV", "Electronics", 2,
     45000.0, "Credit Card", 10.0),

    (113, "Bed", "Furniture", 1,
     28000.0, "UPI", 5.0)
]

discount_df = spark.createDataFrame(
    discount_data,
    discount_schema
)

In [ ]:
discount_df.write \
    .format("delta") \
    .mode("append") \
    .save(delta_path)

In [ ]:
spark.read \
    .format("delta") \
    .load(delta_path) \
    .printSchema()

In [ ]:
spark.read \
    .format("delta") \
    .load(delta_path) \
    .orderBy("SalesID") \
    .show(truncate=False)

In [ ]:
spark.conf.set(
    "spark.databricks.delta.schema.autoMerge.enabled",
    "false"
)

In [ ]:
delta_table = DeltaTable.forPath(
    spark,
    delta_path
)

delta_table.update(
    condition=col("SalesID") == 101,
    set={
        "Price": lit(50000)
    }
)

In [ ]:
delta_table.update(
    condition=col("SalesID") == 102,
    set={
        "Price": lit(23000),
        "Quantity": lit(3),
        "PaymentMode": lit("UPI")
    }
)

In [ ]:
spark.read \
    .format("delta") \
    .load(delta_path) \
    .orderBy("SalesID") \
    .show(truncate=False)

In [ ]:
delta_table.update(
    condition=col("Category") == "Furniture",
    set={
        "Price": col("Price") * 1.10
    }
)

In [ ]:
delta_table.delete(
    condition=col("SalesID") == 105
)

In [ ]:
delta_table.delete(
    condition=(
        (col("Category") == "Electronics") &
        (col("Price") < 1000)
    )
)

In [ ]:
customer_data = [
    (1, "Amit", "Ahmedabad", "Gold"),
    (2, "Neha", "Mumbai", "Silver"),
    (3, "Rahul", "Pune", "Bronze")
]

customer_columns = [
    "CustomerID",
    "CustomerName",
    "City",
    "Membership"
]

customer_df = spark.createDataFrame(
    customer_data,
    customer_columns
)

customer_path = "output/delta/customers"

customer_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(customer_path)

In [ ]:
source_data = [
    (2, "Neha", "Delhi", "Gold"),
    (4, "Priya", "Surat", "Silver")
]

source_df = spark.createDataFrame(
    source_data,
    customer_columns
)

In [ ]:
customer_table = DeltaTable.forPath(
    spark,
    customer_path
)

customer_table.alias("target") \
    .merge(
        source_df.alias("source"),
        "target.CustomerID = source.CustomerID"
    ) \
    .whenMatchedUpdate(
        set={
            "CustomerName": "source.CustomerName",
            "City": "source.City",
            "Membership": "source.Membership"
        }
    ) \
    .whenNotMatchedInsert(
        values={
            "CustomerID": "source.CustomerID",
            "CustomerName": "source.CustomerName",
            "City": "source.City",
            "Membership": "source.Membership"
        }
    ) \
    .execute()

In [ ]:
cdc_data = [
    (1, "Amit", "Vadodara", "Platinum", "U"),
    (3, "Rahul", "Pune", "Bronze", "D"),
    (5, "Karan", "Rajkot", "Silver", "I")
]

cdc_df = spark.createDataFrame(
    cdc_data,
    customer_columns + ["Operation"]
)

In [ ]:
customer_table.alias("target") \
    .merge(
        cdc_df.alias("source"),
        "target.CustomerID = source.CustomerID"
    ) \
    .whenMatchedDelete(
        condition="source.Operation = 'D'"
    ) \
    .whenMatchedUpdate(
        condition="source.Operation = 'U'",
        set={
            "CustomerName": "source.CustomerName",
            "City": "source.City",
            "Membership": "source.Membership"
        }
    ) \
    .whenNotMatchedInsert(
        condition="source.Operation = 'I'",
        values={
            "CustomerID": "source.CustomerID",
            "CustomerName": "source.CustomerName",
            "City": "source.City",
            "Membership": "source.Membership"
        }
    ) \
    .execute()

In [ ]:
delta_table.vacuum()

In [ ]:
delta_table.vacuum(2)